In [1]:
import json
import pandas as pd
import google.generativeai as genai
import time
import re
import random
import uuid
import os
from dotenv import load_dotenv

c:\Users\ThinkPad\Downloads\data-science-main\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ThinkPad\AppData\Local\Temp\ipykernel_16616\471299413.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
# Load file .env yang ada di direktori
load_dotenv()

# Mengambil API key dari environment variables
my_api_key = os.getenv("GEMINI_API_KEY")
new_api_key = os.getenv("NEW_API_KEY")

In [3]:
# # Set up API Key Gemini
# genai.configure(api_key=my_api_key)

# # Kumpulan model flash & lite yang aktif di laptopmu
# MODEL_POOL = [
#     'gemini-2.5-flash-lite',
#     'gemini-2.0-flash',
#     'gemini-flash-latest',
#     'gemini-flash-lite-latest',
#     'gemini-3.1-flash-lite',
#     'gemini-2.0-flash-lite'
# ]

# # Mulai dengan model pertama di dalam list
# current_model_index = 0
# model = genai.GenerativeModel(MODEL_POOL[current_model_index])
# print(f"Mulai menggunakan model utama: {MODEL_POOL[current_model_index]}")

In [4]:
genai.configure(api_key=new_api_key)

MODEL_POOL = [
    'gemini-2.5-flash-lite',
    'gemini-2.0-flash',
    'gemini-flash-latest',
    'gemini-flash-lite-latest',
    'gemini-3.1-flash-lite',
    'gemini-2.0-flash-lite'
]

current_model_index = 0
model = genai.GenerativeModel(MODEL_POOL[current_model_index])

print(f"Sukses ganti akun! Mulai ulang dengan model utama: {MODEL_POOL[current_model_index]}")

Sukses ganti akun! Mulai ulang dengan model utama: gemini-2.5-flash-lite


In [5]:
# 1. Load Data
with open('../data/question_seed.json', 'r') as f:
    question_seeds = json.load(f)

with open('../data/scoring_rubric.json', 'r') as f:
    scoring_rubric = json.load(f)

with open('../data/weakness_taxonomy.json', 'r') as f:
    weakness_taxonomy = json.load(f)

# Buat list berisi semua nama kelemahan untuk dimasukkan ke prompt AI
all_weakness_keys = list(weakness_taxonomy.keys())

In [6]:
# Answer 

general_tags = [
    "context_missing", "contribution_unclear", "tools_missing", "impact_missing", 
    "measurable_result_missing", "star_structure_missing", "role_relevance_low", 
    "technical_detail_missing", "communication_unclear", "self_awareness_missing"
]

role_specific_tags_map = {
    "Cloud Engineer": ["security_awareness_missing", "deployment_missing"],
    "DevOps Engineer": ["security_awareness_missing", "deployment_missing", "testing_coverage_missing"],
    "Manual QA Tester": ["testing_coverage_missing"],
    "QA Automation Engineer": ["testing_coverage_missing", "deployment_missing"],
    "Software Tester": ["testing_coverage_missing"],
    "Mobile Developer": ["mobile_platform_detail_missing"],
    "Android Developer": ["mobile_platform_detail_missing"],
    "iOS Developer": ["mobile_platform_detail_missing"],
    "Cybersecurity Analyst": ["security_awareness_missing"]
}

def generate_synthetic_data(role, competency, question, quality_label):
    
    # 1. Siapkan Allowed Tags
    allowed_tags = general_tags + role_specific_tags_map.get(role, [])
    
    # 2. Randomizer Persona & Gaya STT Mengecoh (Evaluasi Adil)
    weak_personas = [
        "Kandidat menjawab panjang lebar menceritakan teori, tapi sama sekali tidak menceritakan pengalaman pribadinya atau kontribusinya sendiri.",
        "Kandidat sangat gugup, banyak menggunakan filler word (ehh, emm, apa ya), kalimat terputus, dan tidak menyelesaikan jawaban teknisnya.",
        "Menjawab terlalu santai pakai bahasa gaul (kayaknya sih gitu doang, ya biasa aja), sangat pendek, dan tidak memberikan detail tools sama sekali.",
        "Menceritakan masalah kelompok secara detail ('kita/kami'), namun gagal menjelaskan peran spesifik dirinya sendiri (role contribution hilang)."
    ]
    
    avg_personas = [
        "Percaya diri dan menggunakan bahasa interview formal, menceritakan aksinya tapi lupa menyebutkan metrik keberhasilan atau dampak konkret (No impact/No metric).",
        "Menjawab secara runut (struktur oke), menyebutkan tools, tapi solusinya sangat standar dan kurang memiliki akurasi teknis yang mendalam.",
        "Gaya bahasa lisan mengalir natural ('terus...', 'nah abis itu...'), menceritakan konteks dengan baik, tapi tidak menyelesaikan bagian penutup (Result) dengan jelas.",
        "Kandidat menjawab dengan baik di awal, namun di tengah jalan ia terlihat ragu-ragu dan melakukan klarifikasi minor pada jawabannya sendiri."
    ]
    
    # 3. Dynamic Prompt Building
    if quality_label == "Weak":
        flaw = random.choice(weak_personas)
        quality_rules = f"""
        Buat jawaban kandidat yang BURUK/LEMAH (Skor komponen 20-49). Evidence level 1 atau 2.
        Skenario karakteristik jawaban: "{flaw}"
        PENTING: Jangan buat teks yang terlalu mudah ditebak buruk dari kata kasarnya. Buatlah jawaban yang realistis: kadang panjang tapi isinya kosong (omong kosong belaka), tidak menceritakan aksi nyata, atau minim bukti teknis.
        Berdasarkan jawabanmu, pilih semua weakness_tags yang relevan dari list ini: {allowed_tags}.
        """
    elif quality_label == "Average":
        flaw = random.choice(avg_personas)
        quality_rules = f"""
        Buat jawaban kandidat yang CUKUP / BIASA SAJA (Skor komponen 50-74). Evidence level 3 atau 4.
        Skenario karakteristik jawaban: "{flaw}"
        PENTING: Jawaban ini harus bisa 'menipu' model agar tidak langsung dianggap Strong. Buat strukturnya tampak meyakinkan dan menggunakan bahasa interview sehari-hari (kemudian, jadi tuh, nah, dll), tetapi secara substansi hilangkan metrik keberhasilannya (No metrics) atau buat dampaknya kurang terukur.
        Berdasarkan jawabanmu, pilih minimal 2 atau 3 weakness_tags yang paling relevan dari list ini: {allowed_tags}.
        """
    else: # Strong
        quality_rules = f"""
        Buat jawaban kandidat yang SANGAT BAIK/PRO (Skor komponen 75-100). Evidence level 4 atau 5.
        Kandidat menjawab dengan sangat percaya diri, terstruktur penuh dengan STAR Method (Situation, Task, Action, Result), kontribusi personal yang sangat jelas, menggunakan kombinasi tools industri yang tepat, serta menyebutkan metrik angka keberhasilan yang konkret (misal: efisiensi naik 30%, dll).
        Gaya bicara tetap lisan natural interview (bukan kaku seperti membaca buku teks), minim filler words, dan artikulasinya jelas.
        Kolom weakness_tags HARUS dibiarkan kosong [].
        """

    system_prompt = f"""
    Kamu adalah seorang pakar HRD Wawancara Teknis yang bertugas membuat dataset sintetis berkualitas tinggi untuk melatih model klasifikasi. Target aplikasinya akan memproses hasil Speech-to-Text (STT) wawancara kerja.
    Target Role: {role}
    Competency: {competency}
    Pertanyaan: {question}

    TUGAS 1: Buat transkrip STT jawaban kandidat menggunakan sudut pandang orang pertama ("Saya...").
    {quality_rules}

    TUGAS 2: Lakukan self-annotation yang jujur dan ketat terhadap teks jawabanmu sendiri untuk diekstrak menjadi JSON. Jangan overconfident memberikan skor tinggi jika instruksinya meminta label Weak atau Average!
    
    OUTPUT WAJIB berupa JSON dengan struktur persis seperti ini, tanpa markdown/tulisan tambahan:
    {{
        "answer": "Teks transkrip jawaban lisan di sini...",
        "role_relevance": angka_skor_0_sampai_100,
        "star_structure": angka_skor_0_sampai_100,
        "evidence_specificity": angka_skor_0_sampai_100,
        "technical_accuracy": angka_skor_0_sampai_100,
        "communication_clarity": angka_skor_0_sampai_100,
        "self_awareness": angka_skor_0_sampai_100,
        "evidence_level": angka_1_sampai_5,
        "weakness_tags": ["tag1", "tag2"], 
        "has_tool": 1_atau_0,
        "has_metric": 1_atau_0,
        "has_impact": 1_atau_0,
        "has_action": 1_atau_0,
        "has_context": 1_atau_0
    }}
    """
    
    try:
        response = model.generate_content(system_prompt)
        raw_text = response.text.strip()
        
        if raw_text.startswith("```json"):
            raw_text = raw_text[7:-3].strip()
        elif raw_text.startswith("```"):
            raw_text = raw_text[3:-3].strip()
            
        ai_data = json.loads(raw_text)
        return ai_data
    except json.JSONDecodeError:
        print(f"  [!] Teks bukan JSON valid untuk {role} - {quality_label}. Dilewati.")
        return None
    except Exception as e:
        raise e

In [7]:
# Scoring Calculation

def calculate_ds_metrics(ai_data, rubric_weights, taxonomy):
    """
    Fungsi ini mengambil raw data dari AI dan melakukan perhitungan matematis
    dan logical rules sesuai dengan standard scoring kamu.
    """
    if not ai_data:
        return None

    # 1. Hitung Final Score sesuai Bobot (Weights)
    weights = rubric_weights['components']
    weighted_score = (
        ai_data.get('role_relevance', 0) * weights['role_relevance']['weight'] +
        ai_data.get('star_structure', 0) * weights['star_structure']['weight'] +
        ai_data.get('evidence_specificity', 0) * weights['evidence_specificity']['weight'] +
        ai_data.get('technical_accuracy', 0) * weights['technical_accuracy']['weight'] +
        ai_data.get('communication_clarity', 0) * weights['communication_clarity']['weight'] +
        ai_data.get('self_awareness', 0) * weights['self_awareness']['weight']
    )
    
    final_score_0_100 = round(weighted_score, 2)
    final_score_0_1 = round(weighted_score / 100, 4)

    # 2. Logika need_clarification (Sesuai need_clarification_rule di rubricmu)
    weak_tags = ai_data.get('weakness_tags', [])
    need_clarif = False
    if final_score_0_1 < 0.75 or ai_data.get('evidence_level', 1) <= 3 or len(weak_tags) > 0:
        need_clarif = True

    # 3. Ambil clarification_type dari weakness_taxonomy
    clarif_type = "none"
    if need_clarif and len(weak_tags) > 0:
        # Ambil tag pertama sebagai prioritas utama untuk diklarifikasi
        first_tag = weak_tags[0]
        if first_tag in taxonomy:
            clarif_type = taxonomy[first_tag]['clarification_type']
    elif need_clarif:
        # Fallback jika butuh klarifikasi tapi AI tidak memberikan tag
        clarif_type = "general"

    # 4. Hitung panjang kata
    answer_text = ai_data.get('answer', '')
    word_count = len(answer_text.split())

    # Kembalikan data yang sudah lengkap
    return {
        "answer": answer_text,
        "role_relevance": ai_data.get('role_relevance'),
        "star_structure": ai_data.get('star_structure'),
        "evidence_specificity": ai_data.get('evidence_specificity'),
        "technical_accuracy": ai_data.get('technical_accuracy'),
        "communication_clarity": ai_data.get('communication_clarity'),
        "self_awareness": ai_data.get('self_awareness'),
        "evidence_level": ai_data.get('evidence_level'),
        "weakness_tags": ";".join(weak_tags), # Ubah list jadi string dipisah titik koma
        "need_clarification": need_clarif,
        "clarification_type": clarif_type,
        "final_score_0_100": final_score_0_100,
        "final_score_0_1": final_score_0_1,
        "has_tool": ai_data.get('has_tool'),
        "has_metric": ai_data.get('has_metric'),
        "has_impact": ai_data.get('has_impact'),
        "has_action": ai_data.get('has_action'),
        "has_context": ai_data.get('has_context'),
        "answer_length_words": word_count
    }

In [8]:
# Save file 

domain_name = "Information & Technology"
output_file = 'answer_quality_dataset_v2_fixed.csv'

# Urutan kolom sesuai format 
kolom_urut = [
    'sample_id', 'domain', 'role_family', 'target_role', 'competency', 'question', 'answer',
    'role_relevance', 'star_structure', 'evidence_specificity', 'technical_accuracy', 
    'communication_clarity', 'self_awareness', 'evidence_level', 'weakness_tags', 
    'need_clarification', 'clarification_type', 'final_score_0_100', 'final_score_0_1', 
    'quality_label', 'has_tool', 'has_metric', 'has_impact', 'has_action', 'has_context', 
    'answer_length_words'
]

# Ambil total data yang sudah ada di CSV untuk counter yang akurat
current_total_rows = 0
if os.path.exists(output_file):
    try:
        df_existing = pd.read_csv(output_file)
        current_total_rows = len(df_existing)
    except:
        pass

if current_total_rows > 0:
    print(f"Menemukan {current_total_rows} baris data sebelumnya. Melanjutkan proses...")
else:
    pd.DataFrame(columns=kolom_urut).to_csv(output_file, index=False)
    print("Membuat file CSV baru...")

print("Memulai proses generate data target 10.000 baris...")

# TARGET 10 PUTARAN (1.008 data x 10 putaran = 10.080 data total)
for batch_num in range(1, 11):
    
    # Cek ulang jumlah baris sebelum memulai batch baru
    if os.path.exists(output_file):
        try:
            with open(output_file, 'r') as f:
                current_total_rows = sum(1 for line in f) - 1
        except:
            pass
            
    if current_total_rows >= 10000:
        break
        
    print(f"\n🚀 START BATCH KE-{batch_num} DARI TARGET 10 BATCH")
    
    for role_name, questions in question_seeds.items():
        print(f"\nMemproses Role: {role_name} ({len(questions)} pertanyaan) - Batch {batch_num}")
        
        for q_data in questions:
            role_family = q_data.get('role_family')
            competency = q_data.get('competency')
            question_text = q_data.get('question_seed')
            
            for quality in ["Weak", "Average", "Strong"]:
                
                # Cek real-time jumlah baris tiap satu data tersimpan
                if os.path.exists(output_file):
                    try:
                        with open(output_file, 'r') as f:
                            current_total_rows = sum(1 for line in f) - 1
                    except:
                        pass
                
                # JIKA TARGET 10.000 DATA SUDAH TEMBUS, STOP OTOMATIS!
                if current_total_rows >= 10000:
                    print(f"\n[🎉] TARGET SUDAH TEMBUS {current_total_rows} DATA! LOOPING DIHENTIKAN.")
                    break
                
                max_retries = len(MODEL_POOL)
                for attempt in range(max_retries):
                    try:
                        ai_output = generate_synthetic_data(
                            role=role_name, 
                            competency=competency, 
                            question=question_text, 
                            quality_label=quality
                        )
                        
                        if ai_output:
                            calculated_data = calculate_ds_metrics(ai_output, scoring_rubric, weakness_taxonomy)
                            
                            if calculated_data:
                                unique_id = f"AQD-{str(uuid.uuid4().hex)[:6].upper()}"
                                
                                row = {
                                    "sample_id": unique_id,
                                    "domain": domain_name,
                                    "role_family": role_family,
                                    "target_role": role_name,
                                    "competency": competency,
                                    "question": question_text,
                                    **calculated_data,
                                    "quality_label": quality
                                }
                                
                                df_row = pd.DataFrame([row])
                                df_row = df_row[kolom_urut]
                                df_row.to_csv(output_file, mode='a', header=False, index=False)
                                print(f"  [+] Saved ({current_total_rows + 1}/10000): {role_name} | {quality} (via {MODEL_POOL[current_model_index]})")
                        
                        time.sleep(4)
                        break 
                        
                    except Exception as e:
                        if "429" in str(e) or "quota" in str(e).lower():
                            current_model_index += 1
                            if current_model_index >= len(MODEL_POOL):
                                print("\n[🚨] Semua model cadangan sudah terkena limit harian Google! Proses terpaksa berhenti.")
                                raise e
                            print(f"\n[🔄] Model {MODEL_POOL[current_model_index-1]} kena limit harian. Otomatis switch ke: {MODEL_POOL[current_model_index]}")
                            model = genai.GenerativeModel(MODEL_POOL[current_model_index])
                            continue 
                        else:
                            print(f"  [!] Error non-kuota: {e}. Menunggu 10 detik...")
                            time.sleep(10)
            
            if current_total_rows >= 10000: break
        if current_total_rows >= 10000: break
            
print(f"\nProses selesai sepenuhnya! Target 10.000 data berhasil dicapai.")

Menemukan 1590 baris data sebelumnya. Melanjutkan proses...
Memulai proses generate data target 10.000 baris...

🚀 START BATCH KE-1 DARI TARGET 10 BATCH

Memproses Role: Data Analyst (21 pertanyaan) - Batch 1
  [+] Saved (1615/10000): Data Analyst | Weak (via gemini-2.5-flash-lite)
  [+] Saved (1616/10000): Data Analyst | Average (via gemini-2.5-flash-lite)
  [+] Saved (1617/10000): Data Analyst | Strong (via gemini-2.5-flash-lite)
  [+] Saved (1618/10000): Data Analyst | Weak (via gemini-2.5-flash-lite)
  [+] Saved (1619/10000): Data Analyst | Average (via gemini-2.5-flash-lite)
  [+] Saved (1620/10000): Data Analyst | Strong (via gemini-2.5-flash-lite)
  [+] Saved (1621/10000): Data Analyst | Weak (via gemini-2.5-flash-lite)
  [+] Saved (1622/10000): Data Analyst | Average (via gemini-2.5-flash-lite)
  [+] Saved (1623/10000): Data Analyst | Strong (via gemini-2.5-flash-lite)
  [+] Saved (1624/10000): Data Analyst | Weak (via gemini-2.5-flash-lite)
  [+] Saved (1625/10000): Data Analy

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
Please retry in 57.050744058s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerMod

In [ ]:
print("Daftar nama model yang BISA kamu pakai saat ini:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        # Kita hapus 'models/' karena genai.GenerativeModel otomatis menambahkannya
        clean_name = m.name.replace('models/', '')
        print(f"- '{clean_name}'")

Daftar nama model yang BISA kamu pakai saat ini:


DefaultCredentialsError: 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    - Manually pass the key with `genai.configure(api_key=my_api_key)`.
    - Or set up Application Default Credentials, see https://ai.google.dev/gemini-api/docs/oauth for more information.